# Demo del capitolo 2 — audiolab

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


## Demo 6 — La scala temperata

In [ ]:
y, sr = al.load_audio(AUDIO / "C-major-scale.wav", sr=SR)
nomi = ["do", "do#", "re", "re#", "mi", "fa", "fa#", "sol", "sol#", "la", "la#", "si"]
freq_teoriche = al.midi_to_frequency(60) * 2 ** (np.arange(12) / 12)

print(f"{'semitono':9s} {'teorica [Hz]':>13s} {'fattore':>9s}")
for i, nome in enumerate(nomi):
    fattore = freq_teoriche[i] / freq_teoriche[i - 1] if i else float("nan")
    print(f"{nome:9s} {freq_teoriche[i]:13.2f} {fattore:9.4f}")
print("\nfattore atteso per un semitono: 2^(1/12) =", round(2 ** (1 / 12), 4))
print("dopo 12 semitoni:", round(freq_teoriche[11] / freq_teoriche[0] * 2 ** (1 / 12), 3), "(cioè 2 volte do4)")

display(al.safe_audio(y, sr, normalize=False))

# Le frequenze teoriche della scala maggiore: toni e semitoni, non 12 note cromatiche.
passi = np.array([0, 2, 4, 5, 7, 9, 11, 12])
nomi_scala = ["do4", "re4", "mi4", "fa4", "sol4", "la4", "si4", "do5"]
frequenze_scala = al.midi_to_frequency(60) * 2 ** (passi / 12)
fig = go.Figure(go.Scatter(x=nomi_scala, y=frequenze_scala, mode="lines+markers",
                           line=dict(color="#1769C2"), marker=dict(size=8)))
fig.update_yaxes(title_text="frequenza teorica [Hz]")
fig.update_xaxes(title_text="nota")
fig.update_layout(height=340, title="Scala di do maggiore: frequenze teoriche")
fig.show()
# La registrazione completa verrà analizzata nel tempo nella Demo 10.


## Demo 6 ter — Lo spettro della stessa nota su quattro strumenti

In [ ]:
strumenti = [("Piano_C4.wav", "pianoforte"), ("Trumpet_C4.wav", "tromba"),
             ("Violin_C4.wav", "violino"), ("Flute_C4.wav", "flauto")]
fig = make_subplots(rows=2, cols=4, subplot_titles=[n for _, n in strumenti] + [""] * 4)
for k, (file, nome) in enumerate(strumenti):
    y, sr = al.load_audio(AUDIO / file, sr=SR)
    f, amp = al.amplitude_spectrum(y[int(0.5 * sr):int(0.5 * sr) + 8192], sr)
    s = 20 * np.log10(np.maximum(amp, 1e-8))
    fig.add_trace(go.Scatter(x=f, y=amp, mode="lines", line=dict(width=1, color="#1769C2"),
                             showlegend=False), row=1, col=k + 1)
    fig.add_trace(go.Scatter(x=f, y=s, mode="lines", line=dict(width=1, color="#B8860B"),
                             showlegend=False), row=2, col=k + 1)
    picchi = signal.find_peaks(s, height=-50, distance=20)[0]
    primi = [round(f[p]) for p in picchi[:4]]
    print(f"{nome:11s}: picchi a {primi} Hz")
fig.update_xaxes(range=[0, 1000]); fig.update_yaxes(range=[-80, 0], row=2)
fig.update_yaxes(title_text="ampiezza", row=1, col=1); fig.update_yaxes(title_text="livello [dB]", row=2, col=1)
fig.update_layout(height=520, title="Lo spettro della stessa nota su quattro strumenti")
fig.show()


## Demo 7 — Durata osservata e griglia della FFT

In [ ]:
for durata in (0.005, 0.02, 0.1):
    n = int(durata * SR)
    x = np.sin(2 * np.pi * 1037 * np.arange(n) / SR)
    f, amp = al.amplitude_spectrum(x, SR)
    print(f"durata {1000*n/SR:.2f} ms | passo bin {SR/n:.2f} Hz | picco {f[np.argmax(amp)]:.2f} Hz")
n = int(0.02 * SR)
x = np.sin(2 * np.pi * 1037 * np.arange(n) / SR)
fig = go.Figure()
for n_fft in (n, 8 * n):
    f, amp = al.amplitude_spectrum(x, SR, n_fft=n_fft)
    fig.add_trace(go.Scatter(x=f, y=amp, mode="lines+markers", line=dict(width=1),
                             marker=dict(size=3),
                             name=f"FFT {n_fft}: passo {SR / n_fft:.2f} Hz"))
fig.add_vline(x=1037, line=dict(color="gray", dash="dash"), annotation_text="1037 Hz")
fig.update_xaxes(range=[800, 1250], title_text="frequenza [Hz]")
fig.update_yaxes(title_text="ampiezza")
fig.update_layout(height=340, title="Stessi campioni, diversa densità della griglia")
fig.show()


## Demo 8 — Finestre e leakage

In [ ]:
n = int(0.0105 * SR)
x = np.sin(2 * np.pi * 1000 * np.arange(n) / SR)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Lobi laterali", "Lobo principale: dettaglio"))
for nome in ("boxcar", "hann", "hamming", "blackman"):
    f, amp = al.amplitude_spectrum(x, SR, n_fft=32 * n, window=nome)
    db = 20 * np.log10(np.maximum(amp, 1e-8))
    fig.add_trace(go.Scatter(x=f, y=db, mode="lines", line=dict(width=1), name=nome), row=1, col=1)
    fig.add_trace(go.Scatter(x=f, y=db, mode="lines", line=dict(width=1), name=nome,
                             showlegend=False), row=1, col=2)
fig.update_xaxes(range=[0, 5000], row=1, col=1)
fig.update_xaxes(range=[650, 1350], row=1, col=2)
fig.update_yaxes(range=[-100, 5], row=1, col=1); fig.update_yaxes(range=[-50, 5], row=1, col=2)
fig.update_layout(height=380, title="Finestre e leakage")
fig.show()


## Demo 9 — Quanto guadagna la FFT

In [ ]:
rng = np.random.default_rng(0)
print(f"{'N':>6s} {'DFT naive':>12s} {'FFT':>10s} {'guadagno':>10s}")
for N in (256, 512, 1024, 2048):
    x = al.white_noise(N, seed=0)
    M = np.exp(-2j * np.pi * np.outer(np.arange(N), np.arange(N)) / N)
    np.testing.assert_allclose(M @ x, np.fft.fft(x), atol=1e-9)
    tempi_dft, tempi_fft = [], []
    for _ in range(7):
        t0 = time.perf_counter(); M @ x; tempi_dft.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); np.fft.fft(x); tempi_fft.append(time.perf_counter() - t0)
    d, f_ = np.median(tempi_dft) * 1000, np.median(tempi_fft) * 1000
    print(f"{N:6d} {d:10.1f} ms {f_:8.2f} ms {d/f_:9.0f}x")


## Demo 1 — La stessa nota su cinque strumenti

In [ ]:
strumenti = [("Flute_C4.wav", "flauto"), ("Trumpet_C4.wav", "tromba"),
             ("Piano_C4.wav", "pianoforte"), ("Violin_C4.wav", "violino"),
             ("Guitar_C4.wav", "chitarra")]

fig = make_subplots(rows=2, cols=5, subplot_titles=[n for _, n in strumenti] + [""] * 5,
                    vertical_spacing=0.12, horizontal_spacing=0.03)
for k, (file, nome) in enumerate(strumenti):
    y, sr = al.load_audio(AUDIO / file, sr=SR)
    t = np.arange(len(y)) / sr
    breve = t <= 0.2
    fig.add_trace(go.Scatter(x=t[breve], y=y[breve], mode="lines",
                             line=dict(width=1, color="#1769C2"), showlegend=False),
                  row=1, col=k + 1)
    f, s = al.db_spectrum(y, sr)
    fig.add_trace(go.Scatter(x=f, y=s, mode="lines",
                             line=dict(width=1, color="#1769C2"), showlegend=False),
                  row=2, col=k + 1)
    for m in range(1, 9):
        fig.add_vline(x=261.63 * m, line=dict(color="#D9D9D9", width=1), row=2, col=k + 1)
fig.update_xaxes(range=[0, 0.2], row=1)
fig.update_xaxes(range=[0, 4000], row=2)
fig.update_yaxes(range=[-75, 5], row=2)
fig.update_xaxes(title_text="tempo [s]", row=1, col=1)
fig.update_xaxes(title_text="frequenza [Hz]", row=2, col=1)
fig.update_layout(height=560, title="La stessa nota su cinque strumenti: forma d'onda (sopra) e spettro (sotto)")
fig.show()

# Livelli originali delle registrazioni, con pause e dissolvenze anti-click.
clip = [al.fade_edges(al.load_audio(AUDIO / file, sr=SR)[0][:SR], SR) for file, _ in strumenti]
pausa = np.zeros(int(0.2 * SR))
display(al.safe_audio(np.concatenate([parte for x in clip for parte in (x, pausa)]), SR, normalize=False))


## Demo 2 — Tono puro e suono complesso

In [ ]:
f0 = 220.0
t = np.arange(2 * SR) / SR
puro = np.sin(2 * np.pi * f0 * t)
complesso = sum(np.sin(2 * np.pi * k * f0 * t) / k for k in range(1, 7))
puro *= 0.15 / np.sqrt(np.mean(puro ** 2))
complesso *= 0.15 / np.sqrt(np.mean(complesso ** 2))
fig = make_subplots(rows=3, cols=2, subplot_titles=("tono puro", "sei armoniche",
                                          "spettro lineare", "spettro lineare",
                                          "livello [dB]", "livello [dB]"))
for j, (x, titolo) in enumerate([(puro, "tono puro"), (complesso, "sei armoniche")]):
    fig.add_trace(go.Scatter(x=t[:441] * 1000, y=x[:441], mode="lines",
                             line=dict(width=1, color="#1769C2"), showlegend=False), row=1, col=j + 1)
    f, amp = al.amplitude_spectrum(x[SR // 2:SR // 2 + SR], SR, window="boxcar")
    selezione = (f <= 1500) & (amp > 1e-8)
    fig.add_trace(go.Bar(x=f[selezione], y=amp[selezione], marker_color="#1769C2",
                         showlegend=False), row=2, col=j + 1)
    fig.add_trace(go.Scatter(x=f[selezione], y=20 * np.log10(np.maximum(amp[selezione], 1e-8)),
                             mode="lines", line=dict(width=1, color="#B8860B"),
                             showlegend=False), row=3, col=j + 1)
fig.update_xaxes(range=[0, 1500], row=2, col=1); fig.update_xaxes(range=[0, 1500], row=2, col=2)
fig.update_xaxes(range=[0, 1500], row=3, col=1); fig.update_xaxes(range=[0, 1500], row=3, col=2)
fig.update_yaxes(range=[-0.4, 0.4], row=1, col=1)
fig.update_layout(height=760, title="Tono puro e suono complesso: forma d'onda, spettro lineare e in dB")
fig.show()


## Demo 2 bis — Il pitch non è il picco più alto

In [ ]:
f0_mancante = 200.0
t_mancante = np.arange(2 * SR) / SR
armoniche = np.arange(1, 7)
pesi = np.array([0.5, 0.9, 1.0, 0.6, 0.4, 0.3])
componenti = np.sin(2 * np.pi * armoniche[:, None] * f0_mancante * t_mancante)
con_fondamentale = pesi @ componenti
pesi_senza = pesi.copy(); pesi_senza[0] = 0
senza_fondamentale = pesi_senza @ componenti
riferimento_pitch = np.sin(2 * np.pi * f0_mancante * t_mancante)
segnali_pitch = [riferimento_pitch, con_fondamentale, senza_fondamentale]
segnali_pitch = [x * (0.12 / np.sqrt(np.mean(x**2))) for x in segnali_pitch]

fig = make_subplots(rows=1, cols=2, subplot_titles=("con fondamentale", "senza fondamentale"))
for j, x_ in enumerate(segnali_pitch[1:], start=1):
    f, amp = al.amplitude_spectrum(x_[:SR], SR, window="boxcar")
    visibili = (f <= 1400) & (amp > 1e-8)
    fig.add_trace(go.Bar(x=f[visibili], y=amp[visibili], marker_color="#1769C2",
                         showlegend=False), row=1, col=j)
    fig.add_vline(x=f0_mancante, line=dict(color="#D47426", dash="dash"), row=1, col=j)
    fig.update_xaxes(range=[0, 1400], row=1, col=j)
fig.add_annotation(x=0.5, y=1.14, xref="paper", yref="paper", showarrow=False,
                   text="la linea arancione è l'altezza di riferimento (200 Hz)", font=dict(size=12))
fig.update_layout(height=360, title="Il pitch non è il picco più alto")
fig.show()
for nome, x in zip(["riferimento: 200 Hz", "con fondamentale", "senza fondamentale"], segnali_pitch):
    print(nome); display(al.safe_audio(al.fade_edges(x, SR), SR, normalize=False))

# La componente a 200 Hz è davvero assente; quella massima è a 600 Hz.
f, amp = al.amplitude_spectrum(segnali_pitch[2][:SR], SR, window="boxcar")
assert amp[np.argmin(abs(f - 200))] < 1e-10
assert f[np.argmax(amp)] == 600


## Demo 3 — Una campana: spettro inarmonico

In [ ]:
y, sr = al.load_audio(AUDIO / "bell.mp3", sr=SR)
f, s = al.db_spectrum(y, sr, n=16384, start=0.2)

f_lin, amp_lin = al.amplitude_spectrum(y[int(0.2 * sr):int(0.2 * sr) + 16384], sr)
visibile = f_lin <= 5000
fig = make_subplots(rows=1, cols=2, subplot_titles=("ampiezza (lineare)", "livello [dB]"))
fig.add_trace(go.Scatter(x=f_lin[visibile], y=amp_lin[visibile], mode="lines",
                         line=dict(width=1, color="#1769C2"), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=f, y=s, mode="lines", line=dict(width=1, color="#B8860B"),
                         showlegend=False), row=1, col=2)
fig.update_xaxes(range=[0, 5000], title_text="frequenza [Hz]")
fig.update_yaxes(range=[-75, 5], row=1, col=2)
fig.update_layout(height=360, title="Campana: componenti inarmoniche")
fig.show()

picchi = signal.find_peaks(s[:len(s) // 2], height=-45, distance=20)[0]
print("prime componenti rilevate [Hz]:", np.round(f[picchi][:8], 1))

display(al.safe_audio(y, sr, normalize=False))


## Demo 4 — Rumore bianco e rumore rosa

In [ ]:
# Rumori riproducibili; stessa RMS digitale, non necessariamente stessa sonorità.
n = 4 * SR
bianco = al.white_noise(n, seed=42)
rosa = al.pink_noise(n, seed=42)
rumori = {}
for file, x in [("white_noise.wav", bianco), ("pink_noise.wav", rosa)]:
    x -= x.mean()
    rumori[file] = 0.1 * x / np.sqrt(np.mean(x ** 2))
fig = make_subplots(rows=2, cols=2, subplot_titles=("rumore bianco", "rumore rosa",
                                              "rumore bianco [dB]", "rumore rosa [dB]"))
for j, (file, nome) in enumerate([("white_noise.wav", "rumore bianco"),
                                  ("pink_noise.wav", "rumore rosa")]):
    y, sr = rumori[file], SR
    f, psd = signal.welch(y, fs=sr, window="hann", nperseg=min(4096, len(y)), scaling="density")
    fig.add_trace(go.Scatter(x=f[1:], y=np.maximum(psd[1:], 1e-16), mode="lines",
                             line=dict(width=1, color="#1769C2"), showlegend=False), row=1, col=j + 1)
    fig.add_trace(go.Scatter(x=f[1:], y=10 * np.log10(np.maximum(psd[1:], 1e-16)), mode="lines",
                             line=dict(width=1, color="#B8860B"), showlegend=False), row=2, col=j + 1)
fig.update_xaxes(type="log", range=[np.log10(20), np.log10(SR / 2)])
fig.update_yaxes(range=[-100, 0], row=2, col=1); fig.update_yaxes(range=[-100, 0], row=2, col=2)
fig.update_layout(height=560, title="Rumore bianco e rosa: densità di potenza lineare e in dB")
fig.show()

for file, nome in [("white_noise.wav", "bianco"), ("pink_noise.wav", "rosa")]:
    y, sr = rumori[file], SR
    banda = al.analysis.feature.spectral_flatness(y=y, n_fft=2048, hop_length=HOP).mean()
    print(f"  {nome:7s} flatness media: {banda:.3f}")

y_b, sr_b = rumori["white_noise.wav"], SR
y_r, sr_r = rumori["pink_noise.wav"], SR
print("rumore bianco"); display(al.safe_audio(al.fade_edges(y_b[:3 * sr_b], sr_b), sr_b, normalize=False))
print("rumore rosa"); display(al.safe_audio(al.fade_edges(y_r[:3 * sr_r], sr_r), sr_r, normalize=False))


## Demo 5 — L'inviluppo di una nota

In [ ]:
t, nota, inv = al.tone_with_envelope(sr=SR)
rms = al.analysis.feature.rms(y=nota, frame_length=1024, hop_length=256)[0]
tempi = al.analysis.frames_to_time(np.arange(len(rms)), sr=SR, hop_length=256)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Nota sintetica con inviluppo di attacco e decadimento", "RMS a finestre"))
fig.add_trace(go.Scatter(x=t, y=nota, mode="lines", line=dict(width=1, color="#1769C2"),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=inv / np.sqrt(2), mode="lines", line=dict(width=1.2, color="#898989"),
                         name="inviluppo / √2 (RMS locale atteso)"), row=2, col=1)
fig.add_trace(go.Scatter(x=tempi, y=rms, mode="lines", line=dict(width=1.6, color="#005BFF"),
                         name="RMS a finestre"), row=2, col=1)
fig.update_xaxes(title_text="tempo [s]", row=2, col=1)
fig.update_layout(height=460, title="L'inviluppo di una nota")
fig.show()

display(al.safe_audio(nota, SR, normalize=False))


## Demo 10 — Lo spettrogramma della scala

In [ ]:
y, sr = al.load_audio(AUDIO / "C-major-scale.wav", sr=SR)

al.plot_waveform(y, sr, title="Scala di do maggiore: forma d'onda")
al.plot_spectrogram(y, sr, n_fft=N_FFT, hop=HOP, up_to=3000,
                     title="Spettrogramma (asse delle frequenze lineare; colori in dB)")

display(al.safe_audio(y, sr, normalize=False))


## Demo 11 — Finestra e hop: due effetti distinti

In [ ]:
y, sr = al.load_audio(AUDIO / "C-major-scale.wav", sr=SR)
configurazioni = [(512, 256, "finestra corta"), (4096, 256, "finestra lunga"),
                  (2048, 128, "hop breve"), (2048, 1024, "hop lungo")]
spettri = []
for lunghezza, hop, titolo in configurazioni:
    w = signal.get_window("hann", lunghezza, fftbins=True)
    S = np.abs(al.analysis.stft(y, n_fft=4096, win_length=lunghezza,
                           hop_length=hop, window=w, center=True)) / w.sum()
    S[1:-1] *= 2
    spettri.append(S)
riferimento = max(S.max() for S in spettri)
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f"{t0}: L={L0}, H={H0} | {1000 * L0 / sr:.1f} ms osservati; passo {1000 * H0 / sr:.1f} ms"
                                    for L0, H0, t0 in configurazioni])
for k, (S, (lunghezza, hop, titolo)) in enumerate(zip(spettri, configurazioni)):
    db = al.analysis.amplitude_to_db(S, ref=riferimento, top_db=None)
    r, c = divmod(k, 2)
    f = np.arange(S.shape[0]) * sr / 4096
    tempi = np.arange(S.shape[1]) * hop / sr
    fig.add_trace(go.Heatmap(x=tempi, y=f, z=db, colorscale="Magma", zmin=-80, zmax=0,
                             showscale=(k == 3), colorbar=dict(title="dB")), row=r + 1, col=c + 1)
fig.update_yaxes(range=[0, 3000])
fig.update_layout(height=640, title="Finestra e hop: due effetti distinti")
fig.show()


## Demo 12 — Dal segmento alla colonna

In [ ]:
sr_vis = 16000
L_vis, H_vis = 400, 160
t_vis = np.arange(sr_vis) / sr_vis
y_vis = 0.4 * signal.chirp(t_vis, f0=300, f1=2400, t1=1, method="linear")
w_vis = signal.get_window("hann", L_vis, fftbins=True)
D_vis = al.analysis.stft(y_vis, n_fft=L_vis, win_length=L_vis,
                     hop_length=H_vis, window=w_vis, center=False)
f_vis = np.fft.rfftfreq(L_vis, 1 / sr_vis)
frames_vis = [40, 41]
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=("1. Due segmenti sovrapposti", "2. Segmento × finestra Hann",
                                    "3. FFT di ciascun segmento", "4. Ogni spettro diventa una colonna"))
fig.add_trace(go.Scatter(x=t_vis, y=y_vis, mode="lines", line=dict(width=0.7, color="#898989"),
                         showlegend=False), row=1, col=1)
for m, colore in zip(frames_vis, ["#1976B5", "#D47426"]):
    inizio = m * H_vis
    segmento = y_vis[inizio:inizio + L_vis]
    pesato = segmento * w_vis
    coeff = np.fft.rfft(pesato)
    np.testing.assert_allclose(coeff, D_vis[:, m], atol=1e-12)
    fig.add_vrect(x0=inizio / sr_vis, x1=(inizio + L_vis) / sr_vis, fillcolor=colore,
                  opacity=0.25, line_width=0, row=1, col=1)
    fig.add_trace(go.Scatter(x=np.arange(L_vis) / sr_vis * 1000, y=pesato, mode="lines",
                             line=dict(width=1.2, color=colore), name=f"frame {m}"), row=1, col=2)
    fig.add_trace(go.Scatter(x=f_vis, y=abs(coeff), mode="lines", line=dict(width=1.2, color=colore),
                             name=f"colonna {m}", showlegend=False), row=2, col=1)
db_vis = al.analysis.amplitude_to_db(abs(D_vis), ref=np.max)
fig.add_trace(go.Heatmap(x=np.arange(D_vis.shape[1]) * H_vis / sr_vis, y=f_vis, z=db_vis,
                         colorscale="Magma", zmin=-80, zmax=0, showscale=False), row=2, col=2)
for m, colore in zip(frames_vis, ["#1976B5", "#D47426"]):
    fig.add_vline(x=m * H_vis / sr_vis, line=dict(color=colore, width=2), row=2, col=2)
fig.update_xaxes(range=[0.38, 0.46], title_text="tempo nel segnale [s]", row=1, col=1)
fig.update_xaxes(title_text="tempo nel segmento [ms]", row=1, col=2)
fig.update_xaxes(range=[0, 3000], title_text="frequenza [Hz]", row=2, col=1)
fig.update_xaxes(title_text="inizio frame [s]", row=2, col=2)
fig.update_yaxes(range=[0, 3000], row=2, col=2)
fig.update_layout(height=640, title="Dal segmento alla colonna")
fig.show()
print("Forma della matrice (bin, frame):", D_vis.shape)
print("Durata osservata:", 1000*L_vis/sr_vis, "ms; passo:", 1000*H_vis/sr_vis, "ms")


## Demo 13 — Cinque pattern da riconoscere

In [ ]:
sr_pattern = 16000
t_pattern = np.arange(sr_pattern) / sr_pattern
rng_pattern = np.random.default_rng(21)
rumore_pattern = rng_pattern.standard_normal(len(t_pattern))
pattern = [
    ("Tono: una riga", np.sin(2*np.pi*600*t_pattern)),
    ("Nota: più armoniche", sum(np.sin(2*np.pi*250*k*t_pattern)/k for k in range(1, 7))),
    ("Chirp: traccia inclinata", signal.chirp(t_pattern, f0=300, f1=2600, t1=1)),
    ("Colpo: banda verticale", rumore_pattern*np.exp(-0.5*((t_pattern-0.5)/0.003)**2)),
    ("Rumore: energia diffusa", rumore_pattern),
]
pattern = [(nome, 0.4*x/np.max(abs(x))) for nome, x in pattern]
D_pattern = [al.analysis.stft(x, n_fft=512, hop_length=128) for _, x in pattern]
ref_pattern = max(np.max(abs(D)) for D in D_pattern)
fig = make_subplots(rows=2, cols=3, subplot_titles=[n for n, _ in pattern] + [""])
for k, ((nome, _), D) in enumerate(zip(pattern, D_pattern)):
    db = al.analysis.amplitude_to_db(abs(D), ref=ref_pattern, top_db=None)
    r, c = divmod(k, 3)
    fig.add_trace(go.Heatmap(x=np.arange(D.shape[1]) * 128 / sr_pattern,
                             y=np.arange(D.shape[0]) * sr_pattern / 512, z=db,
                             colorscale="Magma", zmin=-70, zmax=0,
                             showscale=(k == 4), colorbar=dict(title="dB")), row=r + 1, col=c + 1)
fig.update_yaxes(range=[0, 4000])
fig.update_layout(height=560, title="Cinque pattern da riconoscere")
fig.show()
for nome, x in pattern:
    print(nome); display(al.safe_audio(al.fade_edges(x, sr_pattern), sr_pattern, normalize=False))


## Demo 14 — Il grafico dipende anche da come lo disegniamo

In [ ]:
y_colori, sr_colori = al.load_audio(AUDIO / "C-major-scale.wav", sr=SR)
A_colori = abs(al.analysis.stft(y_colori, n_fft=2048, hop_length=256))
ref_colori = A_colori.max()
db_colori = al.analysis.amplitude_to_db(A_colori, ref=ref_colori, top_db=None)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Asse lineare", "Asse logaritmico"))
for j, asse in enumerate(["linear", "log"], start=1):
    fig.add_trace(go.Heatmap(x=np.arange(db_colori.shape[1]) * 256 / sr_colori,
                             y=np.arange(db_colori.shape[0]) * sr_colori / 2048, z=db_colori,
                             colorscale="Magma", zmin=-80, zmax=0,
                             showscale=False), row=1, col=j)
    if asse == "log":
        fig.update_yaxes(type="log", range=[np.log10(80), np.log10(8000)], row=1, col=j)
    else:
        fig.update_yaxes(range=[80, 8000], row=1, col=j)
fig.update_layout(height=380, title="Il grafico dipende anche da come lo disegniamo")
fig.show()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Intervallo 40 dB", "Intervallo 80 dB"))
for j, limite in enumerate([-40, -80], start=1):
    fig.add_trace(go.Heatmap(x=np.arange(db_colori.shape[1]) * 256 / sr_colori,
                             y=np.arange(db_colori.shape[0]) * sr_colori / 2048, z=db_colori,
                             colorscale="Magma", zmin=limite, zmax=0,
                             showscale=False), row=1, col=j)
fig.update_yaxes(range=[80, 8000])
fig.update_layout(height=380, title="Quaranta o ottanta dB di intervallo: cambia la lettura")
fig.show()

# Confronto numerico prima del pavimento logaritmico.
A_debole = 0.1 * A_colori
maschera = A_colori > 1e-3 * ref_colori
np.testing.assert_allclose(20*np.log10(A_debole[maschera]/A_colori[maschera]), -20)
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=("originale: riferimento comune", "attenuato di 20 dB: riferimento comune",
                                    "originale: massimo proprio", "attenuato di 20 dB: massimo proprio"))
for riga, riferimento_comune in enumerate([True, False]):
    for colonna, (A, nome) in enumerate([(A_colori, "originale"), (A_debole, "attenuato di 20 dB")]):
        ref = ref_colori if riferimento_comune else A.max()
        db = al.analysis.amplitude_to_db(A, ref=ref, top_db=None)
        fig.add_trace(go.Heatmap(x=np.arange(db.shape[1]) * 256 / sr_colori,
                                 y=np.arange(db.shape[0]) * sr_colori / 2048, z=db,
                                 colorscale="Magma", zmin=-80, zmax=0,
                                 showscale=(riga == 1 and colonna == 1),
                                 colorbar=dict(title="dB")), row=riga + 1, col=colonna + 1)
fig.update_yaxes(range=[80, 8000])
fig.update_layout(height=640, title="Riferimento comune o massimo proprio: due letture diverse")
fig.show()


## Demo 15 — Dalla STFT di nuovo all'audio

In [ ]:
y_ritorno, sr_ritorno = al.load_audio(AUDIO / "C-major-scale.wav", sr=SR)
y_ritorno = y_ritorno.astype(np.float64)
D_ritorno = al.analysis.stft(y_ritorno, n_fft=1024, win_length=1024, hop_length=256,
                         window="hann", center=True)
y_ricostruito = al.analysis.istft(D_ritorno, win_length=1024, hop_length=256, window="hann",
                             center=True, length=len(y_ritorno))
y_fase_zero = al.analysis.istft(abs(D_ritorno).astype(complex), win_length=1024, hop_length=256,
                           window="hann", center=True, length=len(y_ritorno))
np.testing.assert_allclose(y_ricostruito, y_ritorno, atol=1e-10)
print("Errore massimo con STFT complessa:", np.max(abs(y_ricostruito-y_ritorno)))
print("Errore RMS con fase zero:", np.sqrt(np.mean((y_fase_zero-y_ritorno)**2)))
print("Frame centrati:", D_ritorno.shape[1], "attesi:", 1+len(y_ritorno)//256)
assert D_ritorno.shape[1] == 1+len(y_ritorno)//256
D_non_centrata = al.analysis.stft(y_ritorno, n_fft=1024, hop_length=256, center=False)
assert D_non_centrata.shape[1] == 1+(len(y_ritorno)-1024)//256
print("Frame senza padding:", D_non_centrata.shape[1])

inizio = min(int(0.5 * sr_ritorno), len(y_ritorno) // 2)
sezione = slice(inizio, inizio + int(0.04 * sr_ritorno))
t_ritorno = np.arange(len(y_ritorno)) / sr_ritorno

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Ricostruzione fedele: coincide", "Fase azzerata: cambia"))
fig.add_trace(go.Scatter(x=t_ritorno[sezione], y=y_ritorno[sezione], mode="lines",
                         line=dict(width=1.2, color="#1769C2"), name="originale"), row=1, col=1)
fig.add_trace(go.Scatter(x=t_ritorno[sezione], y=y_ricostruito[sezione], mode="lines",
                         line=dict(width=1.2, color="#D47426", dash="dash"),
                         name="ricostruito"), row=1, col=1)
fig.add_trace(go.Scatter(x=t_ritorno[sezione], y=y_ritorno[sezione], mode="lines",
                         line=dict(width=1.2, color="#1769C2"), name="originale", showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=t_ritorno[sezione], y=y_fase_zero[sezione], mode="lines",
                         line=dict(width=1.2, color="#2E8B57"), name="fase azzerata"), row=2, col=1)
fig.update_xaxes(title_text="tempo [s]", row=2, col=1)
fig.update_layout(height=480, title="Dalla STFT di nuovo all'audio")
fig.show()
# Guadagno comune alle tre copie per l'ascolto, senza normalizzazioni separate.
guadagno_ritorno = min(1.0, 0.9/max(np.max(abs(x)) for x in [y_ritorno, y_ricostruito, y_fase_zero]))
for nome, x in [("originale", y_ritorno), ("ricostruito", y_ricostruito), ("fase zero", y_fase_zero)]:
    print(nome); display(al.safe_audio(guadagno_ritorno*x, sr_ritorno, normalize=False))
